## **Exercise 05 : Pandas optimizations**

### **05.1 Read the fines.csv that you saved in the previous exercise**

In [1]:
import pandas as pd
import gc
df = pd.read_csv("../../datasets/fines.csv", index_col=None, delimiter=';')
df

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2.0,3200.0,Ford,Focus,2018
1,E432XX77RUS,1.0,6500.0,Toyota,Camry,2008
2,7184TT36RUS,1.0,2100.0,Ford,Focus,1994
3,X582HE161RUS,2.0,2000.0,Ford,Focus,1987
4,92918M178RUS,1.0,5700.0,Ford,Focus,2000
...,...,...,...,...,...,...
925,a,1.0,100000.0,KAMAZ,49252,2019
926,b,2.0,1000.0,VAZ,Baklajan,2019
927,c,1.0,22000.0,KAMAZ,1984,2019
928,d,2.0,4000.0,VAZ,Lastochka,2019


### **05.2 iterations: in all the following subtasks, you need to calculate fines/refund*year for each row and create a new column with the calculated data and measure the time using the magic command %%timeit in the cell**


In [2]:
%%timeit
# loop: write a function that iterates through the dataframe using for i in range(0, len(df)), iloc and append() to a list, 
# assign the result of the function to a new column in the dataframe1
def by_iloc_append(df):
    new_list: list = []
    for i in range(0, len(df)):
        new_list.append(round(df.iloc[i]['Fines'] / (df.iloc[i]['Refund'] * df.iloc[i]['Year']), 2))
    df['fines/refund*year'] = new_list


by_iloc_append(df)

114 ms ± 3.37 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [3]:
%%timeit
# do it using iterrows()
def by_iterrows(df):    
    new_list: list = []
    for index, row in df.iterrows():
        new_list.append(round(row['Fines'] / (row['Refund'] * row['Year']), 2))
                        
    df['fines/refund*year'] = new_list


by_iterrows(df)

36.4 ms ± 555 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [4]:
%%timeit
# do it using apply() and lambda function
def by_apply(df):    
    df['fines/refund*year'] = df.apply(lambda row: round(row['Fines'] / (row['Refund'] * row['Year']), 2), axis = 1)


by_apply(df)

7.16 ms ± 138 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [5]:
%%timeit
# do it using Series objects from the dataframe
def by_series(df):    
    fines, refund, year = df['Fines'], df['Refund'], df['Year']
    df['fines/refund*year'] = round(fines.div(year.mul(refund)), 2)


by_series(df)

182 μs ± 1.02 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [6]:
%%timeit
# do it as in the previous subtask but with the method .values
def by_series_values(df):    
    fines, refund, year = df['Fines'], df['Refund'], df['Year']
    df['fines/refund*year'] = (fines.values/(year.values*refund.values)).round(2)


by_series_values(df)

81.9 μs ± 350 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [7]:
df

,CarNumber,Refund,Fines,Make,Model,Year,fines/refund*year
0,Y163O8161RUS,2.0,3200.0,Ford,Focus,2018,0.79
1,E432XX77RUS,1.0,6500.0,Toyota,Camry,2008,3.24
2,7184TT36RUS,1.0,2100.0,Ford,Focus,1994,1.05
3,X582HE161RUS,2.0,2000.0,Ford,Focus,1987,0.50
4,92918M178RUS,1.0,5700.0,Ford,Focus,2000,2.85
...,...,...,...,...,...,...,...
925,a,1.0,100000.0,KAMAZ,49252,2019,49.53
926,b,2.0,1000.0,VAZ,Baklajan,2019,0.25
927,c,1.0,22000.0,KAMAZ,1984,2019,10.90
928,d,2.0,4000.0,VAZ,Lastochka,2019,0.99


### **05.3 Indexing: measure the time using the magic command %%timeit in the cell**

In [8]:
%%timeit
# get a row for a specific CarNumber, for example, ’O136HO197RUS’
df.loc[df['CarNumber'] == 'O136HO197RUS']

233 μs ± 5.62 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [9]:
# set the index in your dataframe with CarNumber
df.set_index('CarNumber', inplace=True)

In [10]:
%%timeit
# again, get a row for the same CarNumber
df.loc['O136HO197RUS']

46.6 μs ± 714 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### **05.4 Downcasting:**

In [11]:
# run df.info(memory_usage=’deep’), pay attention to the Dtype and the memory usage
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to e
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Refund             930 non-null    float64
 1   Fines              930 non-null    float64
 2   Make               930 non-null    object 
 3   Model              918 non-null    object 
 4   Year               930 non-null    int64  
 5   fines/refund*year  930 non-null    float64
dtypes: float64(3), int64(1), object(2)
memory usage: 235.9 KB


In [12]:
# make a copy() of your initial dataframe into another dataframe optimized
df_optimized = df.copy(deep=False)

In [13]:
# downcast from float64 to float32 for all columns; downcast int64 to the smallest numerical dtype possible 
for col_name in df_optimized:
    match df_optimized[col_name].dtype:
        case 'float64':
            df_optimized[col_name] = pd.to_numeric(df_optimized[col_name], downcast="float")
        case 'int64':
            df_optimized[col_name] = pd.to_numeric(df_optimized[col_name], downcast="integer")    

In [14]:
# run info(memory_usage='deep') for your new dataframe, pay attention to the Dtype and memory usage
df_optimized.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to e
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Refund             930 non-null    float32
 1   Fines              930 non-null    float32
 2   Make               930 non-null    object 
 3   Model              918 non-null    object 
 4   Year               930 non-null    int16  
 5   fines/refund*year  930 non-null    float32
dtypes: float32(3), int16(1), object(2)
memory usage: 219.6 KB


### **05.5 Categories:**

In [15]:
# change the object type columns to the type category
for col_name in df_optimized:
    match df_optimized[col_name].dtype:
        case 'object':
            df_optimized[col_name] = df_optimized[col_name].astype("category")

In [16]:
# This time, check the memory usage, it probably has decrease of 2-3 times compared to the unitial dataframe
df_optimized.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 930 entries, Y163O8161RUS to e
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   Refund             930 non-null    float32 
 1   Fines              930 non-null    float32 
 2   Make               930 non-null    category
 3   Model              918 non-null    category
 4   Year               930 non-null    int16   
 5   fines/refund*year  930 non-null    float32 
dtypes: category(2), float32(3), int16(1)
memory usage: 111.5 KB


### **05.6 Memory clean:**

In [17]:
# using %reset_selective and the library gc clean the memory of your initial dataframe only
%reset_selective df

Once deleted, variables cannot be recovered. Proceed (y/[n])?   y


In [18]:
gc.collect()

6